In [2]:
from itertools import product
from pathlib import Path
from dotenv import load_dotenv
import os
import sys

ROOT = Path.cwd().resolve().parent
sys.path.append(ROOT.as_posix())
load_dotenv((ROOT / ".env").as_posix())

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
if not WANDB_API_KEY:
    raise ValueError("WANDB_API_KEY is missing from .env")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY

CONFIG = {
    "name": "G_SAE",
    "wandb_project": "g-sae-training-1",
    "wandb_entity": None,
    "wandb_group": "g-sae-celeba_attacked-vgg16-features.29",
    "dataset": "celeba_attacked",
    "model": "vgg16",
    "layer": "features.29",
    "latent_factors": [4.0, 6.0],
    "topk_ratios": [0.025, 0.05, 0.1],
    "direction_source": "decoder",
    "recon_weight": 1.0,
    "cond_weight": 1.0,
    "seed": 23,
    "batch_size": 16,
    "epochs": 100,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "train_split": 0.8,
    "num_workers": 0,
    "max_samples": None,
    "device": "mps",
    "check_val_every_n_epoch": 1,
    "save_optional_encoder_directions": False,
}

LATENT_CACHE_PATH = ROOT / "variables" / CONFIG["dataset"] / CONFIG["model"] / f"{CONFIG['layer']}.pth"
EXPORT_DIR = ROOT / "notebooks" / "checkpoints" / "g_sae_baseline" / "exports"
CHECKPOINT_DIR = ROOT / "notebooks" / "checkpoints" / "g_sae_baseline"
SUGGESTED_BASELINE_PATH = ROOT / "variables" / CONFIG["dataset"] / CONFIG["model"] / CONFIG["layer"] / f"{CONFIG['name']}.pth"

RUN_GRID = [
    {"latent_factor": latent_factor, "topk_ratio": topk_ratio}
    for latent_factor, topk_ratio in product(CONFIG["latent_factors"], CONFIG["topk_ratios"])
]

print(f"Latent cache: {LATENT_CACHE_PATH}")
print(f"Checkpoint root: {CHECKPOINT_DIR}")
print(f"Export dir      : {EXPORT_DIR}")
print(f"Suggested move  : {SUGGESTED_BASELINE_PATH}")
print(f"W&B project : {CONFIG['wandb_project']}")
print("Run grid:")
for run in RUN_GRID:
    print(f"  latent_factor={run['latent_factor']}, topk_ratio={run['topk_ratio']}")


Latent cache: /Users/erogullari/Desktop/Workspace/cav-disentanglement/variables/celeba_attacked/vgg16/features.29.pth
Checkpoint root: /Users/erogullari/Desktop/Workspace/cav-disentanglement/notebooks/checkpoints/g_sae_baseline
Export dir      : /Users/erogullari/Desktop/Workspace/cav-disentanglement/notebooks/checkpoints/g_sae_baseline/exports
Suggested move  : /Users/erogullari/Desktop/Workspace/cav-disentanglement/variables/celeba_attacked/vgg16/features.29/G_SAE.pth
W&B project : g-sae-training-1
Run grid:
  latent_factor=4.0, topk_ratio=0.025
  latent_factor=4.0, topk_ratio=0.05
  latent_factor=4.0, topk_ratio=0.1
  latent_factor=6.0, topk_ratio=0.025
  latent_factor=6.0, topk_ratio=0.05
  latent_factor=6.0, topk_ratio=0.1


# Train Baseline `G_SAE` With Lightning + W&B

This notebook trains the baseline guided sparse autoencoder with `pytorch-lightning` and logs each run to Weights & Biases.

The sweep covers `latent_factor in [4.0, 6.0]` and `topk_ratio in [0.025, 0.05, 0.1]`. Each run is exported with a parameter-specific filename under `notebooks/checkpoints/g_sae_baseline/exports`.

The notebook does not auto-copy anything into `variables`. Pick the exported `.pth` you want from `notebooks/checkpoints`, then move it into `variables/{dataset}/{model}/{layer}` yourself.


In [ ]:
from typing import Any

import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback, LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
import wandb

from cav_models import G_SAE

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("medium")


def format_decimal(value: float, min_decimals: int = 1, max_decimals: int = 3) -> str:
    formatted = f"{float(value):.{max_decimals}f}".rstrip("0").rstrip(".")
    if "." not in formatted:
        formatted = f"{formatted}." + ("0" * min_decimals)
    else:
        decimals = len(formatted.split(".", 1)[1])
        if decimals < min_decimals:
            formatted = formatted + ("0" * (min_decimals - decimals))
    return formatted


def make_run_name(base_name: str, latent_factor: float, topk_ratio: float) -> str:
    latent_factor_str = format_decimal(latent_factor, min_decimals=1, max_decimals=1)
    topk_ratio_str = format_decimal(topk_ratio, min_decimals=1, max_decimals=3)
    return f"{base_name}-latent_factor={latent_factor_str}-topk_ratio={topk_ratio_str}"


def resolve_trainer_config(device_name: str) -> dict[str, Any]:
    requested = str(device_name).lower()
    if requested in {"auto", "cuda", "gpu"} and torch.cuda.is_available():
        return {"accelerator": "gpu", "devices": 1, "device_label": "cuda"}
    if requested in {"auto", "mps"} and torch.backends.mps.is_available():
        return {"accelerator": "mps", "devices": 1, "device_label": "mps"}
    return {"accelerator": "cpu", "devices": 1, "device_label": "cpu"}


def normalize_cavs_and_bias(
    cavs: torch.Tensor,
    bias: torch.Tensor,
    eps: float = 1e-12,
) -> tuple[torch.Tensor, torch.Tensor]:
    norms = torch.norm(cavs, dim=1, keepdim=True).clamp_min(eps)
    cavs_norm = cavs / norms
    bias_norm = bias / norms.squeeze(1)
    return cavs_norm, bias_norm


def get_concept_directions(model: G_SAE, source: str = "decoder") -> tuple[torch.Tensor, torch.Tensor]:
    if source == "decoder":
        cavs = model.decoder.weight[:, : model.n_concepts].T.detach().cpu().clone()
        bias = torch.zeros(model.n_concepts, dtype=cavs.dtype)
        return cavs, bias

    if source == "encoder":
        cavs = model.encoder.weight[: model.n_concepts, :].detach().cpu().clone()
        if model.encoder.bias is None:
            bias = torch.zeros(model.n_concepts, dtype=cavs.dtype)
        else:
            bias = model.encoder.bias[: model.n_concepts].detach().cpu().clone()
        return cavs, bias

    raise ValueError(f"Unknown direction source: {source}")


def scalarize_metrics(metrics: dict[str, Any]) -> dict[str, Any]:
    scalarized: dict[str, Any] = {}
    for key, value in metrics.items():
        if isinstance(value, torch.Tensor):
            if value.numel() == 1:
                scalarized[key] = float(value.detach().cpu())
            else:
                scalarized[key] = value.detach().cpu().tolist()
        else:
            scalarized[key] = value
    return scalarized


def build_dataloaders(
    train_dataset,
    val_dataset,
    batch_size: int,
    num_workers: int,
    seed: int,
    pin_memory: bool,
) -> tuple[DataLoader, DataLoader]:
    loader_kwargs = {
        "batch_size": int(batch_size),
        "num_workers": int(num_workers),
        "pin_memory": bool(pin_memory),
        "persistent_workers": bool(num_workers > 0),
        "drop_last": False,
    }
    train_generator = torch.Generator().manual_seed(int(seed))
    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        generator=train_generator,
        **loader_kwargs,
    )
    val_loader = DataLoader(
        val_dataset,
        shuffle=False,
        **loader_kwargs,
    )
    return train_loader, val_loader


class EpochMetricsRecorder(Callback):
    def __init__(self) -> None:
        super().__init__()
        self.history: list[dict[str, float]] = []

    def on_validation_epoch_end(self, trainer, pl_module) -> None:
        if trainer.sanity_checking:
            return

        metric_names = [
            "train_total_loss",
            "train_recon_loss",
            "train_cond_loss",
            "train_latent_density",
            "train_active_latents",
            "val_total_loss",
            "val_recon_loss",
            "val_cond_loss",
            "val_latent_density",
            "val_active_latents",
        ]
        epoch_metrics = {"epoch": int(trainer.current_epoch)}
        for metric_name in metric_names:
            metric_value = trainer.callback_metrics.get(metric_name)
            if metric_value is not None:
                epoch_metrics[metric_name] = float(metric_value.detach().cpu())
        self.history.append(epoch_metrics)


def build_export_payload(
    model: G_SAE,
    run_config: dict[str, Any],
    dataset_metadata: dict[str, int],
    history: list[dict[str, float]],
    best_metrics: dict[str, Any],
) -> dict[str, Any]:
    source = str(run_config["direction_source"])
    cavs_unnorm, bias_unnorm = get_concept_directions(model, source=source)
    cavs_norm, bias_norm = normalize_cavs_and_bias(cavs_unnorm, bias_unnorm)

    metadata = {
        "name": str(CONFIG["name"]),
        "run_name": str(run_config["run_name"]),
        "dataset": str(CONFIG["dataset"]),
        "model": str(CONFIG["model"]),
        "layer": str(CONFIG["layer"]),
        "direction_source": source,
        "latent_factor": float(run_config["latent_factor"]),
        "topk_ratio": float(run_config["topk_ratio"]),
        "topk": int(model.topk),
        "n_samples": int(dataset_metadata["n_samples"]),
        "train_size": int(dataset_metadata["train_size"]),
        "val_size": int(dataset_metadata["val_size"]),
        "n_features": int(model.n_features),
        "n_concepts": int(model.n_concepts),
        "n_latents": int(model.n_latents),
        "seed": int(run_config["seed"]),
        "batch_size": int(run_config["batch_size"]),
        "epochs": int(run_config["epochs"]),
        "learning_rate": float(run_config["learning_rate"]),
        "weight_decay": float(run_config["weight_decay"]),
        "recon_weight": float(run_config["recon_weight"]),
        "cond_weight": float(run_config["cond_weight"]),
        "baseline_definition": "guided_sae_alpha_0",
        "latent_cache_path": str(run_config["latent_cache_path"]),
        "export_path": str(run_config["export_path"]),
        "suggested_baseline_path": str(run_config["suggested_baseline_path"]),
        "best_checkpoint_path": str(run_config.get("best_checkpoint_path") or ""),
        "wandb_project": run_config["wandb_project"],
        "wandb_entity": run_config["wandb_entity"],
        "wandb_group": run_config["wandb_group"],
        "wandb_run_id": run_config.get("wandb_run_id"),
        "wandb_run_url": run_config.get("wandb_run_url"),
        "best_metrics": best_metrics,
        "history": history,
    }

    if bool(CONFIG["save_optional_encoder_directions"]):
        enc_cavs_unnorm, enc_bias_unnorm = get_concept_directions(model, source="encoder")
        enc_cavs_norm, enc_bias_norm = normalize_cavs_and_bias(enc_cavs_unnorm, enc_bias_unnorm)
        metadata["optional_encoder_export"] = {
            "available": True,
            "direction_source": "encoder",
            "entries": {
                "normalized": {
                    "cavs": enc_cavs_norm,
                    "bias": enc_bias_norm,
                },
                "unnormalized": {
                    "cavs": enc_cavs_unnorm,
                    "bias": enc_bias_unnorm,
                },
            },
        }
    else:
        metadata["optional_encoder_export"] = {"available": False}

    return {
        "type": str(CONFIG["name"]),
        "entries": {
            "normalized": {
                "cavs": cavs_norm,
                "bias": bias_norm,
            },
            "unnormalized": {
                "cavs": cavs_unnorm,
                "bias": bias_unnorm,
            },
        },
        "metadata": metadata,
        "state_dict": model.state_dict(),
    }


In [ ]:
assert LATENT_CACHE_PATH.exists(), f"Latent cache not found: {LATENT_CACHE_PATH}"

latent_payload = torch.load(LATENT_CACHE_PATH, map_location="cpu", weights_only=True)
encs = latent_payload["encs"].float()
labels = latent_payload["labels"].float().clamp(min=0)

if CONFIG["max_samples"] is not None:
    max_samples = int(CONFIG["max_samples"])
    encs = encs[:max_samples]
    labels = labels[:max_samples]

n_samples, n_features = encs.shape
n_concepts = labels.shape[1]

dataset = TensorDataset(encs, labels)
train_size = int(len(dataset) * float(CONFIG["train_split"]))
val_size = len(dataset) - train_size
if train_size == 0 or val_size == 0:
    raise ValueError(
        f"Invalid split for {len(dataset)} samples. train_size={train_size}, val_size={val_size}."
    )

split_generator = torch.Generator().manual_seed(int(CONFIG["seed"]))
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=split_generator)

DATASET_METADATA = {
    "n_samples": int(n_samples),
    "train_size": int(train_size),
    "val_size": int(val_size),
}
TRAINER_CONFIG = resolve_trainer_config(str(CONFIG["device"]))

print(f"encs   shape: {tuple(encs.shape)}")
print(f"labels shape: {tuple(labels.shape)}")
print(f"Train size: {train_size}")
print(f"Val size  : {val_size}")
print(
    "Trainer device:",
    TRAINER_CONFIG["device_label"],
    f"(accelerator={TRAINER_CONFIG['accelerator']}, devices={TRAINER_CONFIG['devices']})",
)


In [ ]:
class GSAELightningModule(pl.LightningModule):
    def __init__(self, config: dict[str, Any], n_concepts: int, n_features: int) -> None:
        super().__init__()
        config = dict(config)
        self.save_hyperparameters(
            {
                "config": config,
                "n_concepts": int(n_concepts),
                "n_features": int(n_features),
            }
        )
        self.config = config
        self.model = G_SAE(
            n_concepts=int(n_concepts),
            n_features=int(n_features),
            device="cpu",
            latent_factor=float(config["latent_factor"]),
            topk_ratio=float(config["topk_ratio"]),
            recon_weight=float(config["recon_weight"]),
            cond_weight=float(config["cond_weight"]),
            direction_source=str(config["direction_source"]),
        )

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.model(x)

    def _shared_step(self, batch, stage: str) -> torch.Tensor:
        x, y = batch
        _, latents, recons = self.model(x)

        recon_loss = self.model._normalized_recon_mse(recons, x)
        cond_features = latents[:, : self.model.n_concepts]
        cond_loss = F.binary_cross_entropy(cond_features, y)
        total_loss = (
            float(self.config["recon_weight"]) * recon_loss
            + float(self.config["cond_weight"]) * cond_loss
        )

        latent_active_mask = latents > 0
        latent_density = latent_active_mask.float().mean()
        active_latents = latent_active_mask.float().sum(dim=1).mean()

        log_kwargs = {
            "on_step": False,
            "on_epoch": True,
            "logger": True,
            "prog_bar": stage == "val",
            "batch_size": x.shape[0],
        }
        self.log(f"{stage}_total_loss", total_loss, **log_kwargs)
        self.log(f"{stage}_recon_loss", recon_loss, **log_kwargs)
        self.log(f"{stage}_cond_loss", cond_loss, **log_kwargs)
        self.log(f"{stage}_latent_density", latent_density, **log_kwargs)
        self.log(f"{stage}_active_latents", active_latents, **log_kwargs)

        return total_loss

    def training_step(self, batch, batch_idx) -> torch.Tensor:
        return self._shared_step(batch, stage="train")

    def validation_step(self, batch, batch_idx) -> torch.Tensor:
        return self._shared_step(batch, stage="val")

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=float(self.config["learning_rate"]),
            weight_decay=float(self.config["weight_decay"]),
        )
        return optimizer


In [ ]:
wandb.login(key=WANDB_API_KEY)

RESULTS = []
BEST_RUN = None
pin_memory = TRAINER_CONFIG["device_label"] == "cuda"

for sweep_index, sweep_params in enumerate(RUN_GRID, start=1):
    pl.seed_everything(int(CONFIG["seed"]), workers=True)

    run_name = make_run_name(
        CONFIG["name"],
        latent_factor=float(sweep_params["latent_factor"]),
        topk_ratio=float(sweep_params["topk_ratio"]),
    )
    export_path = EXPORT_DIR / f"{run_name}.pth"
    checkpoint_dir = CHECKPOINT_DIR / run_name

    run_config = {
        key: value
        for key, value in CONFIG.items()
        if key not in {"latent_factors", "topk_ratios"}
    }
    run_config.update(sweep_params)
    run_config.update(
        {
            "run_name": run_name,
            "export_path": export_path.as_posix(),
            "suggested_baseline_path": SUGGESTED_BASELINE_PATH.as_posix(),
            "checkpoint_dir": checkpoint_dir.as_posix(),
            "latent_cache_path": LATENT_CACHE_PATH.as_posix(),
        }
    )

    train_loader, val_loader = build_dataloaders(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        batch_size=int(CONFIG["batch_size"]),
        num_workers=int(CONFIG["num_workers"]),
        seed=int(CONFIG["seed"]),
        pin_memory=pin_memory,
    )

    history_callback = EpochMetricsRecorder()
    checkpoint_callback = ModelCheckpoint(
        dirpath=checkpoint_dir,
        filename=run_name + "-{epoch:02d}-{val_total_loss:.4f}",
        monitor="val_total_loss",
        mode="min",
        save_top_k=1,
        save_last=False,
        auto_insert_metric_name=False,
        verbose=True,
    )
    lr_monitor = LearningRateMonitor(logging_interval="epoch")
    wandb_logger = WandbLogger(
        project=CONFIG["wandb_project"],
        entity=CONFIG["wandb_entity"],
        name=run_name,
        group=CONFIG["wandb_group"],
        job_type="train",
        save_dir=ROOT.as_posix(),
        log_model=False,
        reinit=True,
        tags=["G_SAE", CONFIG["dataset"], CONFIG["model"], CONFIG["layer"]],
    )

    lightning_model = GSAELightningModule(
        config=run_config,
        n_concepts=n_concepts,
        n_features=n_features,
    )
    run_hparams = {
        **run_config,
        **DATASET_METADATA,
        "n_features": int(n_features),
        "n_concepts": int(n_concepts),
        "n_latents": int(lightning_model.model.n_latents),
        "topk": int(lightning_model.model.topk),
        "accelerator": TRAINER_CONFIG["accelerator"],
        "devices": TRAINER_CONFIG["devices"],
        "grid_size": len(RUN_GRID),
    }
    wandb_logger.log_hyperparams(run_hparams)

    trainer = pl.Trainer(
        max_epochs=int(CONFIG["epochs"]),
        accelerator=TRAINER_CONFIG["accelerator"],
        devices=TRAINER_CONFIG["devices"],
        logger=wandb_logger,
        callbacks=[checkpoint_callback, lr_monitor, history_callback],
        deterministic=True,
        log_every_n_steps=1,
        enable_progress_bar=True,
        enable_model_summary=True,
        check_val_every_n_epoch=int(CONFIG["check_val_every_n_epoch"]),
        default_root_dir=ROOT.as_posix(),
    )

    try:
        trainer.fit(lightning_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

        best_model_path = checkpoint_callback.best_model_path
        if best_model_path:
            best_module = GSAELightningModule.load_from_checkpoint(
                best_model_path,
                config=run_config,
                n_concepts=n_concepts,
                n_features=n_features,
            )
        else:
            best_module = lightning_model

        best_val_metrics_raw = trainer.validate(best_module, dataloaders=val_loader, verbose=False)[0]
        best_val_metrics = scalarize_metrics(best_val_metrics_raw)
        best_score = checkpoint_callback.best_model_score
        best_val_total_loss = (
            float(best_score.detach().cpu())
            if best_score is not None
            else float(best_val_metrics["val_total_loss"])
        )

        run_config["best_checkpoint_path"] = best_model_path
        run_config["wandb_run_id"] = wandb_logger.experiment.id
        run_config["wandb_run_url"] = wandb_logger.experiment.url

        payload = build_export_payload(
            model=best_module.model.to("cpu"),
            run_config=run_config,
            dataset_metadata=DATASET_METADATA,
            history=history_callback.history,
            best_metrics=best_val_metrics,
        )
        export_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(payload, export_path)

        wandb_logger.experiment.summary["export_path"] = export_path.as_posix()
        wandb_logger.experiment.summary["best_checkpoint_path"] = best_model_path
        wandb_logger.experiment.summary["n_latents"] = int(best_module.model.n_latents)
        wandb_logger.experiment.summary["topk"] = int(best_module.model.topk)
        for metric_name, metric_value in best_val_metrics.items():
            wandb_logger.experiment.summary[f"best_{metric_name}"] = metric_value

        result = {
            "run_name": run_name,
            "latent_factor": float(sweep_params["latent_factor"]),
            "topk_ratio": float(sweep_params["topk_ratio"]),
            "n_latents": int(best_module.model.n_latents),
            "topk": int(best_module.model.topk),
            "best_val_total_loss": float(best_val_total_loss),
            "best_val_recon_loss": float(best_val_metrics.get("val_recon_loss", float("nan"))),
            "best_val_cond_loss": float(best_val_metrics.get("val_cond_loss", float("nan"))),
            "best_val_latent_density": float(best_val_metrics.get("val_latent_density", float("nan"))),
            "best_val_active_latents": float(best_val_metrics.get("val_active_latents", float("nan"))),
            "export_path": export_path.as_posix(),
            "checkpoint_path": best_model_path,
            "wandb_run_url": wandb_logger.experiment.url,
        }
        RESULTS.append(result)
        print(
            f"[{sweep_index}/{len(RUN_GRID)}] "
            f"Saved {run_name} to {export_path} | "
            f"best val_total_loss={result['best_val_total_loss']:.6f}"
        )
    finally:
        wandb.finish()


In [ ]:
assert RESULTS, "No sweep runs were recorded."

RESULTS = sorted(RESULTS, key=lambda item: item["best_val_total_loss"])
BEST_RUN = RESULTS[0]
BEST_EXPORT_PATH = Path(BEST_RUN["export_path"])

print("Sweep summary (sorted by val_total_loss):")
for rank, result in enumerate(RESULTS, start=1):
    print(
        f"{rank:02d}. {result['run_name']} | "
        f"val_total={result['best_val_total_loss']:.6f} | "
        f"val_recon={result['best_val_recon_loss']:.6f} | "
        f"val_cond={result['best_val_cond_loss']:.6f} | "
        f"n_latents={result['n_latents']} | topk={result['topk']}"
    )

print()
print(f"Best run              : {BEST_RUN['run_name']}")
print(f"Best run export       : {BEST_EXPORT_PATH}")
print(f"Suggested move target : {SUGGESTED_BASELINE_PATH}")
print(f"Best W&B run URL      : {BEST_RUN['wandb_run_url']}")


In [ ]:
assert BEST_RUN is not None, "Best run was not populated."

for result in RESULTS:
    export_path = Path(result["export_path"])
    assert export_path.exists(), f"Missing export file: {export_path}"
    reloaded = torch.load(export_path, map_location="cpu", weights_only=True)

    assert reloaded["type"] == "G_SAE", f"Unexpected type in {export_path}: {reloaded['type']}"
    cavs = reloaded["entries"]["normalized"]["cavs"]
    bias = reloaded["entries"]["normalized"]["bias"]
    assert cavs.shape == (n_concepts, n_features), (
        f"Expected cavs shape {(n_concepts, n_features)}, got {tuple(cavs.shape)} for {export_path}"
    )
    assert bias.shape[0] == n_concepts, (
        f"Expected bias length {n_concepts}, got {bias.shape[0]} for {export_path}"
    )
    norms = torch.norm(cavs, dim=1)
    assert torch.allclose(norms, torch.ones_like(norms), atol=1e-4), (
        f"Normalized cavs are not unit norm for {export_path}"
    )

print("Validated all checkpoint exports.")
print(f"Validated {len(RESULTS)} parameter-specific exports.")
